In [9]:
# import necessary libraries
import requests
import json
import pandas as pd
import os
import time

In [10]:
def fetch_all_pages(url_template: str, label: str, pause: float = 0.15) -> list:
    """Paginate through all cursor pages for a given URL template."""
    results = []
    cursor  = '*'

    while cursor:
        url  = url_template.format(cursor) 
        resp = requests.get(url)

        if resp.status_code == 429:
            print(f'[{label}] rate limited, backing off 5s...')
            time.sleep(5)
            continue  # retry same cursor

        resp.raise_for_status()
        data = resp.json()

        page_results = data['results']
        results     += page_results
        cursor       = data['meta']['next_cursor']

        print(f'[{label}] fetched {len(page_results)} results | '
              f'total so far: {len(results)} | next cursor: {cursor}')

        time.sleep(pause)  # small gap between requests to stay under burst limit

    return results

## QUERY 1 - Defence & dual-use technologies 

In [11]:
import requests

BASE    = 'https://api.openalex.org/works'
FILTERS = 'institutions.id:I18014758,publication_year:2020-2024'
PARAMS  = 'include_xpac=true&per_page=200&cursor={}'

SEARCH_STEMMED = (
    '"defense technology" OR "dual-use technology" OR "security technology" OR '
    '"military technology" OR "national security" OR "security innovation" OR '
    '"civil-military" OR "aerospace engineering" OR "space domain awareness" OR '
    '"autonomous system" OR "unmanned system" OR "robotic system" OR '
    '"surveillance system" OR "situational awareness" OR "sensor fusion" OR '
    '"remote sensing" OR "radar system" OR "infrared sensing" OR '
    '"hyperspectral sensing" OR "sonar system" OR "optical system" OR '
    '"secure communication" OR "tactical communication" OR '
    '"mission-critical system" OR "resilient communication" OR '
    '"navigation resilience" OR "cyber defence" OR "electronic warfare" OR '
    '"electromagnetic spectrum" OR "signals intelligence" OR '
    '"information operation" OR "advanced material" OR "structural material" OR '
    '"lightweight material" OR "high-performance material" OR '
    '"ballistic material" OR "armour material" OR "blast-resistant material" OR '
    '"stealth material" OR "explosive detection" OR "hazard detection" OR '
    '"threat detection" OR wargaming OR "operational analysis" OR '
    '"decision support system" OR "mission planning" OR '
    '"human systems integration" OR "human performance" OR '
    '"hypersonic system" OR "advanced propulsion" OR "rocket propulsion" OR '
    '"arms control" OR "export control" OR '
    '"dual-use research governance" OR "civil protection" OR '
    '"emergency response system"'
)

SEARCH_EXACT = 'space* OR operat* OR ethic*'

URL_STEMMED = f'{BASE}?search={SEARCH_STEMMED}&filter={FILTERS}&{PARAMS}'
URL_EXACT   = f'{BASE}?search.exact={SEARCH_EXACT}&filter={FILTERS}&{PARAMS}'


# --- Run both queries ---
raw_stemmed = fetch_all_pages(URL_STEMMED, label='stemmed')
raw_exact   = fetch_all_pages(URL_EXACT,   label='exact')

# --- Deduplicate by OpenAlex work ID ---
seen = {}
for work in raw_stemmed + raw_exact:
    seen[work['id']] = work          # last-write-wins; IDs are stable so order doesn't matter

full_results = list(seen.values())

print(f'\nDone. {len(raw_stemmed)} stemmed + {len(raw_exact)} exact → '
      f'{len(full_results)} unique works after deduplication.')

[stemmed] fetched 200 results | total so far: 200 | next cursor: IlsyLjI0MzU2ODQsIDE2MTQ3Mjk2MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzEzNTAzMDc3NiddIg==
[stemmed] fetched 200 results | total so far: 400 | next cursor: IlswLjkxNzU5MDQsIDE2NjQyMzY4MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XNDI5NzIwNjc2OCddIg==
[stemmed] fetched 200 results | total so far: 600 | next cursor: IlswLjI0NDA1OTI4LCAxNjI1MDExMjAwMDAwLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzMxODEzNDE1MDAnXSI=
[stemmed] fetched 78 results | total so far: 678 | next cursor: None
[exact] fetched 200 results | total so far: 200 | next cursor: Ils0LjUzMTEyOSwgMTU4NzQyNzIwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1czMDMwODQ1MjQyJ10i
[exact] fetched 200 results | total so far: 400 | next cursor: IlszLjQxNTQ3NTgsIDE2NjY0ODMyMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XNDMxMjQzNzk0NiddIg==
[exact] fetched 200 results | total so far: 600 | next cursor: IlsyLjc5MTI4OCwgMTY0NzM4ODgwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0MjIxMDI2MzYxJ10i
[exact] fetched 200 r

## QUERY 2 - Manufacturing & Advanced Materials

In [12]:
BASE    = 'https://api.openalex.org/works'
FILTERS = 'institutions.id:I18014758,publication_year:2020-2024'
PARAMS  = 'include_xpac=true&per_page=200&cursor={}'

SEARCH_STEMMED_2 = (
    '"manufacturing engineering" OR "industrial automation" OR '
    '"predictive maintenance" OR "powder bed fusion" OR '
    '"direct energy deposition" OR "material science" OR '
    '"material design" OR "material processing" OR "material synthesis" OR '
    '"material characterization" OR "material performance" OR '
    '"material mechanic" OR "surface engineering" OR '
    '"laser surface processing" OR "plasma surface treatment" OR '
    '"microfabrication" OR "nanofabrication" OR "semiconductor processing" OR '
    '"thin film deposition" OR "fatigue behavior" OR "fatigue life" OR '
    '"failure analysis" OR "computational material" OR '
    '"integrated computational material engineering" OR '
    '"closed-loop manufacturing" OR "design for manufacturability" OR '
    '"design for disassembly" OR "resource-efficient manufacturing" OR '
    '"cyber-physical production system" OR "machine vision for manufacturing" OR '
    '"quality inspection in manufacturing" OR "robot-assisted manufacturing" OR '
    '"smart manufacturing" OR "digital manufacturing" OR '
    '"intelligent manufacturing" OR "sustainable manufacturing"'
)

WILDCARD_TERMS_2 = [
    'manufactur*',
    '"advanced manufactur*"',
    '"manufacturing system*"',
    '"manufacturing process*"',
    '"manufacturing technolog*"',
    '"smart manufactur*"',
    '"digital manufactur*"',
    '"intelligent manufactur*"',
    '"cyber-physical production system*"',
    '"manufactur* automation"',
    '"industrial robotic*"',
    '"additive manufactur*"',
    '"3D print*"',
    '"metal additive manufactur*"',
    '"polymer additive manufactur*"',
    '"precision machin*"',
    '"ultra-precision manufactur*"',
    '"material* scienc*"',
    '"advanced material*"',
    '"functional material*"',
    '"structural material*"',
    '"engineering material*"',
    '"material* property*"',
    '"material* mechanic*"',
    '"composite material*"',
    '"fiber-reinforced composite*"',
    '"lightweight material*"',
    '"high-performance material*"',
    '"high-entropy alloy*"',
    '"metal alloy*"',
    '"aluminium alloy*"',
    '"magnesium alloy*"',
    '"superalloy*"',
    '"corrosion-resistant material*"',
    '"nanomaterial*"',
    '"nanostructured material*"',
    '"thin film*"',
    '"coating technolog*"',
    '"fracture mechanic*"',
    '"damage mechanism*"',
    '"material* informatic*"',
    '"multi-scale modell*"',
    '"remanufactur*"',
]

URL_STEMMED_2 = f'{BASE}?search={SEARCH_STEMMED_2}&filter={FILTERS}&{PARAMS}'

# --- Run stemmed query ---
raw_stemmed_2 = fetch_all_pages(URL_STEMMED_2, label='stemmed')

# --- Run one request per wildcard term and collect all results ---
raw_exact_2 = []
for term in WILDCARD_TERMS_2:
    url_template = f'{BASE}?search.exact={term}&filter={FILTERS}&{PARAMS}'
    raw_exact_2 += fetch_all_pages(url_template, label=f'exact:{term}')

# --- Deduplicate across all results ---
seen_2 = {}
for work in raw_stemmed_2 + raw_exact_2:
    seen_2[work['id']] = work

full_results_2 = list(seen_2.values())

print(f'\nDone. {len(raw_stemmed_2)} stemmed + {len(raw_exact_2)} exact (pre-dedup) → '
      f'{len(full_results_2)} unique works after deduplication.')

[stemmed] fetched 200 results | total so far: 200 | next cursor: IlswLjA2MTI4NjM3NSwgMTczNTYwMzIwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0NDA2NDgxMzQyJ10i
[stemmed] fetched 6 results | total so far: 206 | next cursor: None
[exact:manufactur*] fetched 200 results | total so far: 200 | next cursor: IlswLjQ2ODMzMDAzLCAxNjE3MTQ4ODAwMDAwLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzMxNTExNzM5NTMnXSI=
[exact:manufactur*] fetched 200 results | total so far: 400 | next cursor: IlswLjI1NjE1NTI4LCAxNTg2MzA0MDAwMDAwLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzMwMTU1NDk0OTInXSI=
[exact:manufactur*] fetched 200 results | total so far: 600 | next cursor: IlswLjIsIDE2ODI4OTkyMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XNDM3NjEzMTIzMCddIg==
[exact:manufactur*] fetched 200 results | total so far: 800 | next cursor: IlswLjE1LCAxNzE1ODE3NjAwMDAwLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzQzOTY5NTg2NzknXSI=
[exact:manufactur*] fetched 200 results | total so far: 1000 | next cursor: IlswLjEsIDE3MTI3MDcyMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XND

## QUERY 3 - Democratic & Community Resilience

In [14]:
SEARCH_STEMMED_3 = (
    'democracy OR "democratic governance" OR "democratic backsliding" OR '
    '"democratic quality" OR "civic engagement" OR "civic participation" OR '
    '"political participation" OR "deliberative democracy" OR '
    '"participatory governance" OR "public participation" OR '
    '"citizen engagement" OR "social cohesion" OR "social capital" OR '
    '"community capacity" OR "collective action" OR "collective efficacy" OR '
    '"community development" OR "place-based development" OR "local community" OR '
    '"neighbourhood" OR "public trust" OR "political trust" OR '
    '"institutional trust" OR "trust in institution" OR legitimacy OR '
    '"governance legitimacy" OR governance OR "public governance" OR '
    '"institutional capacity" OR "policy capacity" OR "public administration" OR '
    '"state capacity" OR "multi-level governance" OR "urban governance" OR '
    '"local governance" OR "adaptive governance" OR "crisis governance" OR '
    'misinformation OR disinformation OR "media literacy" OR '
    '"political communication" OR "digital civic engagement" OR '
    '"human rights" OR "minority rights" OR "indigenous governance" OR '
    '"access to justice" OR "restorative justice" OR peacebuilding OR '
    '"social inequality" OR "spatial inequality" OR "housing insecurity" OR '
    'homelessness OR "inclusive governance" OR "public opinion" OR '
    '"public consultation" OR "community-based participatory research" OR '
    '"participatory action research"'
)

WILDCARD_TERMS_3 = [
    '"democratic institution*"',
    '"democratic process*"',
    '"democratic resilien*"',
    '"community resilien*"',
    '"social resilien*"',
    '"institutional resilien*"',
    '"policy resilien*"',
    '"migration polic*"',
    '"integration polic*"',
    '"anti-racist polic*"',
]

URL_STEMMED_3 = f'{BASE}?search={SEARCH_STEMMED_3}&filter={FILTERS}&{PARAMS}'

# --- Run stemmed query ---
raw_stemmed_3 = fetch_all_pages(URL_STEMMED_3, label='stemmed')

# --- Run one request per wildcard term and collect all results ---
raw_exact_3 = []
for term in WILDCARD_TERMS_3:
    url_template = f'{BASE}?search.exact={term}&filter={FILTERS}&{PARAMS}'
    raw_exact_3 += fetch_all_pages(url_template, label=f'exact:{term}')

# --- Deduplicate across all results ---
seen_3 = {}
for work in raw_stemmed_3 + raw_exact_3:
    seen_3[work['id']] = work

full_results_3 = list(seen_3.values())

print(f'\nDone. {len(raw_stemmed_3)} stemmed + {len(raw_exact_3)} exact (pre-dedup) → '
      f'{len(full_results_3)} unique works after deduplication.')

[stemmed] fetched 200 results | total so far: 200 | next cursor: IlsyNy4xNzc3MDgsIDE2NjgzODQwMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XNDMwOTAwMTQ4MiddIg==
[stemmed] fetched 200 results | total so far: 400 | next cursor: IlsxNC42MTkyNTQsIDE2MDM5Mjk2MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzEyNDk4MTQ0MiddIg==
[stemmed] fetched 200 results | total so far: 600 | next cursor: Ils4LjUzNDI3MywgMTY1NTk0MjQwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0MjgzMzExMzkyJ10i
[stemmed] fetched 200 results | total so far: 800 | next cursor: Ils0LjQ5MzgxODgsIDE2NTMzNTA0MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XNDI4MTM5MDg2MyddIg==
[stemmed] fetched 200 results | total so far: 1000 | next cursor: IlsyLjg5NDkxMjcsIDE2MDE0MjQwMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzA5MDc4ODkyMSddIg==
[stemmed] fetched 200 results | total so far: 1200 | next cursor: IlsyLjA3Njg5MDIsIDE1Nzc4MzY4MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzAwNjAyNTg1NyddIg==
[stemmed] fetched 200 results | total so far: 1400 | next cursor: IlsxLjU0MjE0

KeyboardInterrupt: 

## 2024 High Impact Pubs

In [ ]:
# initialize the API URL
# the cursor allows us to pull multiple pages of results to get all SFU works in one call. 
# NOTE: SFU's institutional id in OpenAlex is I18014f758

url_with_cursor = 'https://api.openalex.org/works?page=1&filter=publication_year:2024,authorships.institutions.lineage:i18014758&cursor={}'
cursor = '*'

# empty containter to store the results
full_results_4 = []

# loop through pages
while cursor:
    
    # set cursor value and request page from OpenAlex
    url = url_with_cursor.format(cursor)
    
    #print("\n" + url)
    page_with_results = requests.get(url)
    page_with_results = page_with_results.json()
    
    # loop through partial list of results
    results = page_with_results['results']
    
    full_results_4 += results

    # update cursor to meta.next_cursor
    cursor = page_with_results['meta']['next_cursor']
    #print("next cursor is: ", cursor)

## SAVE ALL

In [ ]:
#pd.DataFrame(full_results_1).to_csv('./cerc_results/query1_V2.csv', index = False)
#pd.DataFrame(full_results_2).to_csv('./cerc_results/query2_V2.csv', index = False)
#pd.DataFrame(full_results_3).to_csv('./cerc_results/query3_V2.csv', index = False)


In [15]:
pd.DataFrame(full_results).to_csv('./query1_RERUN.csv', index = False)
pd.DataFrame(full_results_2).to_csv('./query2_RERUN.csv', index = False)
pd.DataFrame(full_results_3).to_csv('./query3_RERUN.csv', index = False)

NameError: name 'full_results_3' is not defined

In [ ]:
pd.DataFrame(full_results_4).to_csv('./cerc_results/works2024.csv', index = False)

In [ ]:
len(full_results_3)

2177